# MCP — Model Context Protocol

MCP lets an LLM call tools exposed by a server. The LLM sees the tool definitions, decides when to call them, and the server executes them.

```
User question → LLM decides to call a tool → MCP Client sends request → MCP Server runs it → result back to LLM
```

In [ ]:
# --- Read the MCP server source code ---
# MCP servers run as separate processes (not inside notebooks),
# but we can read and study the code here.

with open("grocery_mcp_server.py") as f:
    print(f.read())

## How the Server Works

| Part | What it does |
|------|-------------|
| `Server("grocery-tools")` | Creates the server with a name clients see |
| `@server.list_tools()` | Tells the LLM what tools exist (name, description, parameters) |
| `@server.call_tool()` | Executes the tool when the LLM calls it |
| `stdio_server()` | Communicates via stdin/stdout (client launches server as a subprocess) |

The three tools:
- **`search_products`** — keyword search with optional category filter
- **`get_nutrition`** — nutrition facts by product ID
- **`check_stock`** — stock level at a specific store

## Configuring a Client

To connect this server to Claude Desktop, add to `claude_desktop_config.json`:

```json
{
  "mcpServers": {
    "grocery": {
      "command": "python",
      "args": ["/full/path/to/grocery_mcp_server.py"]
    }
  }
}
```

Config file location:
- **macOS:** `~/Library/Application Support/Claude/claude_desktop_config.json`
- **Windows:** `%APPDATA%\Claude\claude_desktop_config.json`

After saving, restart the client. The tools appear automatically.

In [ ]:
# --- Simulate what the MCP server does when tools are called ---
# We can import and call the handler logic directly to see the output.

import importlib, sys, os
sys.path.insert(0, os.getcwd())

import grocery_mcp_server as mcp

# Simulate: search_products
print("=== search_products('organic') ===")
query = "organic"
matches = []
for pid, p in mcp.PRODUCTS.items():
    if query in p["name"].lower() or query in p["description"].lower():
        matches.append(f"  {pid}: {p['name']} — ${p['price']:.2f}")
for m in matches:
    print(m)

In [ ]:
# Simulate: get_nutrition
print("=== get_nutrition('egg-003') ===")
p = mcp.PRODUCTS["egg-003"]
print(f"  {p['name']}")
print(f"  Calories: {p['calories']} | Protein: {p['protein_g']}g | Fat: {p['fat_g']}g")
print(f"  Organic: {p['organic']} | Price: ${p['price']:.2f}")

In [ ]:
# Simulate: check_stock across all stores
print("=== check_stock('bread-001') ===")
pid = "bread-001"
name = mcp.PRODUCTS[pid]["name"]
for store, inventory in mcp.STOCK.items():
    qty = inventory.get(pid, 0)
    status = f"IN STOCK ({qty})" if qty > 0 else "OUT OF STOCK"
    print(f"  {store:>10}: {status}")

## Key Takeaways

1. **MCP is a protocol** — one standard for any LLM to call any tool
2. **Servers expose tools** — search, calculate, check stock, anything
3. **The LLM decides** when to call tools based on descriptions you write
4. **Servers run as subprocesses** — the client launches them, communicates via stdio
5. **You control access** — only expose what the LLM should use

Further reading: [modelcontextprotocol.io](https://modelcontextprotocol.io)